# Day 17 (Notebook 2) - Building a Mini Semantic Search Engine

This notebook builds a complete retrieval pipeline **without an LLM**.

Pipeline:

```
User Query
     ↓
Sentence Embedding
     ↓
Qdrant
     ↓
Top-K Chunks
     ↓
Display Results
```

The objective is to understand retrieval before adding generation (LLM) on Day 18.


In [1]:
# Uncomment if required
# !pip install sentence-transformers qdrant-client pandas ipywidgets

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
import pandas as pd
from ipywidgets import interact_manual, Text, IntSlider
import os
from dotenv import load_dotenv

# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("HF_TOKEN"):
    raise ValueError("HF_TOKEN not found. Please check your .env file.")

model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient(host="localhost", port=6333)

print("Connected to Qdrant")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to Qdrant


## Step 1 - Sample Enterprise Knowledge Base

In [2]:
knowledge = [
("HR","Employees receive 18 days of annual leave every year."),
("HR","Medical leave allows up to 10 days annually."),
("HR","Work from home is permitted three days each week."),
("Travel","International travel requires Vice President approval."),
("Travel","Hotel reimbursement follows company policy."),
("Travel","Taxi reimbursement is allowed for official travel."),
("Finance","Expense claims should be submitted within 30 days."),
("Finance","Salary is credited on the last working day."),
("Finance","Travel advances must be settled within 15 days."),
("IT","VPN access is mandatory for remote work."),
("IT","Company laptops are replaced every four years."),
("IT","Multi-factor authentication is compulsory."),
("Engineering","Python is preferred for AI development."),
("Engineering","Docker is used for containerized deployment."),
("Engineering","GitHub is used for source control."),
("Facilities","Office timings are 9 AM to 6 PM."),
("Facilities","Parking is allocated based on availability."),
("Security","Visitors must carry a valid ID."),
("Security","Passwords should never be shared."),
("Security","USB drives are disabled by default.")
]

df = pd.DataFrame(knowledge, columns=["Department","Chunk"])
df


,Department,Chunk
0,HR,Employees receive 18 days of annual leave ever...
1,HR,Medical leave allows up to 10 days annually.
2,HR,Work from home is permitted three days each week.
3,Travel,International travel requires Vice President a...
4,Travel,Hotel reimbursement follows company policy.
5,Travel,Taxi reimbursement is allowed for official tra...
6,Finance,Expense claims should be submitted within 30 d...
7,Finance,Salary is credited on the last working day.
8,Finance,Travel advances must be settled within 15 days.
9,IT,VPN access is mandatory for remote work.


## Step 2 - Create Embeddings

In [3]:
texts = df["Chunk"].tolist()
embeddings = model.encode(texts)

print("Embedding Shape:", embeddings.shape)


Embedding Shape: (20, 384)


## Step 3 - Create / Recreate Qdrant Collection

In [4]:
COLLECTION = "mini_enterprise_rag"

try:
    client.delete_collection(COLLECTION)
except:
    pass

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

points=[]

for idx,row in df.iterrows():
    points.append(
        PointStruct(
            id=int(idx),
            vector=embeddings[idx].tolist(),
            payload={
                "department":row["Department"],
                "text":row["Chunk"]
            }
        )
    )

client.upsert(collection_name=COLLECTION, points=points)

print(f"Inserted {len(points)} chunks.")


Inserted 20 chunks.


## Step 4 - Interactive Semantic Search

In [5]:
def semantic_search(query, top_k=5):

    query_vector = model.encode(query).tolist()

    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_k
    ).points

    print("="*80)
    print("USER QUERY:", query)
    print("="*80)

    for rank, r in enumerate(results,1):
        print(f"Rank #{rank}")
        print(f"Score      : {r.score:.4f}")
        print(f"Department : {r.payload['department']}")
        print(f"Chunk      : {r.payload['text']}")
        print("-"*80)

interact_manual(
    semantic_search,
    query=Text(
        value="How many vacation days do employees receive?",
        description="Query:"
    ),
    top_k=IntSlider(value=5,min=1,max=10)
)


interactive(children=(Text(value='How many vacation days do employees receive?', continuous_update=False, desc…

<function __main__.semantic_search(query, top_k=5)>

## Step 5 - Suggested Queries

In [6]:
queries = [
"How many annual leave days do employees receive?",
"Can I work from home?",
"Who approves international travel?",
"When are salaries credited?",
"How often are laptops replaced?",
"What programming language is preferred for AI?",
"What are the office timings?",
"Can I share my password?"
]

for q in queries:
    print("\n")
    semantic_search(q,3)




USER QUERY: How many annual leave days do employees receive?
Rank #1
Score      : 0.8697
Department : HR
Chunk      : Employees receive 18 days of annual leave every year.
--------------------------------------------------------------------------------
Rank #2
Score      : 0.6499
Department : HR
Chunk      : Medical leave allows up to 10 days annually.
--------------------------------------------------------------------------------
Rank #3
Score      : 0.5097
Department : Finance
Chunk      : Salary is credited on the last working day.
--------------------------------------------------------------------------------


USER QUERY: Can I work from home?
Rank #1
Score      : 0.6837
Department : HR
Chunk      : Work from home is permitted three days each week.
--------------------------------------------------------------------------------
Rank #2
Score      : 0.3969
Department : IT
Chunk      : VPN access is mandatory for remote work.
-----------------------------------------------------

# Discussion

Observe that:

- The query rarely contains the exact wording from the stored chunk.
- Retrieval is based on **semantic similarity**, not keyword matching.
- No LLM is involved—Qdrant simply returns the most relevant chunks.

---

# Bridge to Day 18

Tomorrow we'll extend the pipeline:

```
User Question
      ↓
Embedding
      ↓
Qdrant
      ↓
Top-K Chunks
      ↓
Prompt Construction
      ↓
LLM
      ↓
Grounded Answer
```

Day 17 answers **"Which chunks are relevant?"**

Day 18 answers **"How do we use those chunks to generate a grounded response?"**
